<a href="https://colab.research.google.com/github/Jirtus-sanasam/MLP-Diabetes/blob/main/diabetesall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
df = pd.read_csv('/content/diabetes_data2.csv')

In [5]:
cols_invalid_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age']
cols_valid_zero = ['Pregnancies', 'DiabetesPedigreeFunction']

In [6]:
df[cols_invalid_zero] = df[cols_invalid_zero].replace(0, np.nan)

In [7]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(random_state=42, max_iter=10)
df[cols_invalid_zero] = imputer.fit_transform(df[cols_invalid_zero])

In [8]:
X = df.drop('Outcome', axis=1)
Y = df['Outcome']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [27]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [48]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    penalty='l2',            # 'l1', 'l2', 'elasticnet', None
    C=1.0,                   # Inverse regularization strength (smaller = stronger reg)
    solver='lbfgs',          # 'lbfgs', 'liblinear', 'saga', 'newton-cg', 'sag'
    max_iter=1000,
    class_weight='balanced', # None or 'balanced' (useful for imbalanced data)
    l1_ratio=0.5,            # Only used when penalty='elasticnet' (0=l2, 1=l1)
    tol=1e-4,                # Tolerance for stopping criteria
    random_state=42
)
lr.fit(X_train_sc, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


LogisticRegression(class_weight='balanced', l1_ratio=0.5, max_iter=1000,
                   random_state=42)

In [49]:
y_pred      = lr.predict(X_test_sc)
y_pred_prob = lr.predict_proba(X_test_sc)[:, 1]

In [50]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)

In [51]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             roc_curve)
print("=" * 50)
print("       LOGISTIC REGRESSION - RESULTS")
print("=" * 50)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  AUC-ROC   : {auc:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))

       LOGISTIC REGRESSION - RESULTS
  Accuracy  : 0.7338
  Precision : 0.6032
  Recall    : 0.7037
  F1-Score  : 0.6496
  AUC-ROC   : 0.8120

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.82      0.75      0.79       100
    Diabetes       0.60      0.70      0.65        54

    accuracy                           0.73       154
   macro avg       0.71      0.73      0.72       154
weighted avg       0.75      0.73      0.74       154



In [52]:
svm = SVC(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    class_weight='balanced',
    probability=True,        # Required for predict_proba & AUC-ROC
    tol=1e-3,
    random_state=42
)

svm.fit(X_train_sc, y_train)

SVC(class_weight='balanced', probability=True, random_state=42)

In [53]:
y_pred      = svm.predict(X_test_sc)
y_pred_prob = svm.predict_proba(X_test_sc)[:, 1]

In [54]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)
print("=" * 50)
print("       SUPPORT VECTOR MACHINE - RESULTS")
print("=" * 50)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  AUC-ROC   : {auc:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))

       SUPPORT VECTOR MACHINE - RESULTS
  Accuracy  : 0.7403
  Precision : 0.6029
  Recall    : 0.7593
  F1-Score  : 0.6721
  AUC-ROC   : 0.8087

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.85      0.73      0.78       100
    Diabetes       0.60      0.76      0.67        54

    accuracy                           0.74       154
   macro avg       0.73      0.74      0.73       154
weighted avg       0.76      0.74      0.75       154



In [56]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=100,        # Number of trees (more = better but slower)
    max_depth=None,          # Max depth of each tree (None = unlimited)
    min_samples_split=2,     # Min samples to split a node
    min_samples_leaf=1,      # Min samples at a leaf node
    max_features='sqrt',     # 'sqrt', 'log2', int, float
    bootstrap=True,          # Whether to use bootstrap samples
    class_weight='balanced', # Handles imbalance
    max_samples=None,        # If bootstrap=True, how many samples per tree
    oob_score=True,          # Use out-of-bag samples to estimate accuracy
    n_jobs=-1,               # Use all CPU cores
    random_state=42
)
rf.fit(X_train, y_train)

print(f"  OOB Score : {rf.oob_score_:.4f}")

  OOB Score : 0.7476


In [57]:
y_pred      = rf.predict(X_test)
y_pred_prob = rf.predict_proba(X_test)[:, 1]

In [59]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)
print("=" * 50)
print("         RANDOM FOREST - RESULTS")
print("=" * 50)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  AUC-ROC   : {auc:.4f}")
print(f"  OOB Score : {rf.oob_score_:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))

         RANDOM FOREST - RESULTS
  Accuracy  : 0.7403
  Precision : 0.6522
  Recall    : 0.5556
  F1-Score  : 0.6000
  AUC-ROC   : 0.8262
  OOB Score : 0.7476

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.78      0.84      0.81       100
    Diabetes       0.65      0.56      0.60        54

    accuracy                           0.74       154
   macro avg       0.71      0.70      0.70       154
weighted avg       0.73      0.74      0.73       154



In [62]:
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    scale_pos_weight=1.8,       # ~ratio of neg/pos for diabetes dataset
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)
xgb.fit(
    X_train_sc, y_train,
    eval_set=[(X_train_sc, y_train), (X_test_sc, y_test)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)

In [63]:
y_pred      = xgb.predict(X_test)
y_pred_prob = xgb.predict_proba(X_test)[:, 1]

# ============================================================
# METRICS
# ============================================================
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)

In [64]:
print("=" * 50)
print("           XGBOOST - RESULTS")
print("=" * 50)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  AUC-ROC   : {auc:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))

           XGBOOST - RESULTS
  Accuracy  : 0.3506
  Precision : 0.3506
  Recall    : 1.0000
  F1-Score  : 0.5192
  AUC-ROC   : 0.4406

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.00      0.00      0.00       100
    Diabetes       0.35      1.00      0.52        54

    accuracy                           0.35       154
   macro avg       0.18      0.50      0.26       154
weighted avg       0.12      0.35      0.18       154



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
